In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import f_oneway

# ── 1. Load raw tables ────────────────────────────────────────────────────────
transactions = pd.read_csv('transactions.csv')
# columns: customer_id, order_id, order_date, product_id, category,
#          quantity, unit_price, discount_pct, channel, is_returned

customers = pd.read_csv('customers.csv')
# columns: customer_id, join_date, city, loyalty_tier

products = pd.read_csv('products.csv')
# columns: product_id, category, is_premium

# ── 2. Feature engineering — collapse to 1 row per customer ──────────────────
today = pd.Timestamp('2025-01-01')
txn = transactions.merge(products[['product_id','is_premium']], on='product_id')

features = txn.groupby('customer_id').agg(

    # Value dimension
    total_revenue       = ('unit_price', lambda x: (x * txn.loc[x.index,'quantity']).sum()),
    avg_order_value     = ('order_id',   lambda x: txn.loc[x.index,'unit_price'].sum() / x.nunique()),

    # Frequency dimension
    order_count         = ('order_id',   'nunique'),
    category_breadth    = ('category',   'nunique'),  # how many diff categories bought

    # Recency dimension
    days_since_last     = ('order_date', lambda x: (today - pd.to_datetime(x).max()).days),

    # Product mix dimension
    premium_ratio       = ('is_premium', 'mean'),     # share of premium purchases

    # Price sensitivity dimension
    avg_discount_pct    = ('discount_pct', 'mean'),
    promo_purchase_rate = ('discount_pct', lambda x: (x > 0).mean()),

    # Channel dimension
    online_ratio        = ('channel',    lambda x: (x == 'online').mean()),

).reset_index()

# Merge in customer-level attributes
features = features.merge(customers[['customer_id','loyalty_tier']], on='customer_id')

# ── 3. Handle skewed distributions (spend and frequency are usually right-skewed)
skewed_cols = ['total_revenue', 'avg_order_value', 'order_count', 'days_since_last']
pt = PowerTransformer(method='yeo-johnson')  # handles zeros better than log
features[skewed_cols] = pt.fit_transform(features[skewed_cols])

# ── 4. Filter variables ───────────────────────────────────────────────────────
cluster_vars = [
    'total_revenue', 'avg_order_value',   # value
    'order_count', 'category_breadth',    # frequency & breadth
    'days_since_last',                    # recency
    'premium_ratio',                      # product preference
    'promo_purchase_rate',                # price sensitivity
    'online_ratio'                        # channel preference
]

X = features[cluster_vars].dropna()

# Check variance — drop near-zero variance
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.01)
selector.fit(X)
low_var = [cluster_vars[i] for i,v in enumerate(selector.get_support()) if not v]
print(f"Low variance variables (consider dropping): {low_var}")

# Check correlations
corr = pd.DataFrame(X).corr().abs()
high_corr = [(corr.columns[i], corr.columns[j])
             for i in range(len(corr.columns))
             for j in range(i+1, len(corr.columns))
             if corr.iloc[i,j] > 0.85]
print(f"Highly correlated pairs: {high_corr}")

# ── 5. Standardize ────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── 6. Find optimal k ─────────────────────────────────────────────────────────
results = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X_scaled)
    results.append({
        'k': k,
        'wcss': km.inertia_,
        'silhouette': silhouette_score(X_scaled, labels, sample_size=5000)
    })

results_df = pd.DataFrame(results)
print(results_df)

# ── 7. Fit final model ────────────────────────────────────────────────────────
optimal_k = 4  # adjust based on elbow + silhouette + business sense
km_final = KMeans(n_clusters=optimal_k, n_init=30, random_state=42)
features.loc[X.index, 'segment'] = km_final.fit_predict(X_scaled)

# ── 8. Profile segments ───────────────────────────────────────────────────────
# Use ORIGINAL (un-scaled) values for profiling — much more readable
original_vars = [
    'total_revenue', 'avg_order_value', 'order_count',
    'category_breadth', 'days_since_last', 'premium_ratio',
    'promo_purchase_rate', 'online_ratio'
]
# Re-merge with original values
raw_features = transactions.groupby('customer_id').agg(...).reset_index()  # same agg, no transform
raw_features['segment'] = features['segment']

profile = raw_features.groupby('segment')[original_vars].agg(['mean','median'])
print(profile.T)  # transposed for readability

# ── 9. Validate: are segments statistically distinct? ────────────────────────
# ANOVA: if p < 0.05, the variable meaningfully separates segments
for var in original_vars:
    groups = [raw_features[raw_features['segment']==s][var].dropna()
              for s in raw_features['segment'].unique()]
    f, p = f_oneway(*groups)
    print(f"{var}: F={f:.1f}, p={p:.4f} {'✓' if p < 0.05 else '✗ CHECK THIS'}")

# ── 10. Score new customers into existing segments ────────────────────────────
def score_customer(new_customer_dict):
    row = pd.DataFrame([new_customer_dict])[cluster_vars]
    row_transformed = pt.transform(row[skewed_cols].values.reshape(1,-1))
    # ... apply full pipeline
    row_scaled = scaler.transform(row)
    return km_final.predict(row_scaled)[0]